# Hari 001 — Return harian & kumulatif

| Hari | Tanggal | Jenis | Topik |
|------|---------|-------|-------|
| 001 | 2026-11-19 | `quant` | Return harian & kumulatif |

> Bagian dari **100QB** — 100 Days of Quant + Kebun Simulasi.

## Konsep

**Simple return** (return sederhana) mengukur perubahan persentase harga dari satu periode ke periode berikutnya:

$$R_t = \frac{P_t - P_{t-1}}{P_{t-1}} = \frac{P_t}{P_{t-1}} - 1$$

**Return kumulatif** mengukur total pertumbuhan nilai investasi dari awal periode hingga waktu $t$ dengan memperhitungkan efek *compounding* (bunga majemuk):

$$R_{\text{cum}, t} = \prod_{i=1}^{t} (1 + R_i) - 1 = \frac{P_t}{P_0} - 1$$

**Kenapa dipakai:**
- Harga saham mentah sulit dibandingkan antar aset (misalnya harga dalam ribuan rupiah vs ratusan dolar); return menormalkan skala sehingga performa aset dapat dibandingkan secara adil.
- Return harian bersifat *scale-free* dan menjadi dasar perhitungan risiko (volatilitas), *Sharpe ratio*, serta evaluasi portofolio.

## Setup

In [ ]:
import sys
from pathlib import Path

# Cari root repo supaya package `qb100` di src/ bisa di-import dari mana pun
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "qb100").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qb100 import data, finance as fin, plotting, simulation as sim

plotting.set_style()
rng = sim.get_rng(42)

TICKERS = ["BBCA.JK", "^JKSE", "AAPL"]
START_DATE = "2024-01-01"
END_DATE = "2024-12-31"


## Data

Kita mengambil harga penutupan (*Close*) untuk tiga aset: `BBCA.JK` (Bank Central Asia), `^JKSE` (IHSG), dan `AAPL` (Apple) menggunakan `qb100.data.get_close`. Bila koneksi internet/API `yfinance` gagal atau *offline*, kode otomatis jatuh balik ke data harga simulasi menggunakan `qb100.simulation.random_walk`.

In [ ]:
try:
    prices = data.get_close(TICKERS, start=START_DATE, end=END_DATE)
    prices = prices.ffill().dropna()
    data_source = "yfinance (cache/online)"
except Exception as exc:
    print(f"Gagal mengunduh dari yfinance ({exc}); memakai data simulasi.")
    dates = pd.bdate_range(START_DATE, END_DATE)
    n_steps = len(dates) - 1
    log_paths = sim.random_walk(
        n_steps=n_steps,
        n_paths=len(TICKERS),
        step_std=0.014,
        drift=0.0004,
        start=0.0,
        seed=rng,
    )
    base_prices = np.array([9500.0, 7200.0, 185.0])
    prices = pd.DataFrame(base_prices * np.exp(log_paths), index=dates, columns=TICKERS)
    data_source = "simulasi (random_walk)"

print(f"Sumber data: {data_source} | Baris: {len(prices)}")
prices.head()

## Implementasi

In [ ]:
# 1. Hitung simple return harian menggunakan qb100.finance.simple_returns
daily_ret = fin.simple_returns(prices)
print("5 hari pertama return harian (%):")
(daily_ret * 100).round(3).head()

In [ ]:
# 2. Hitung return kumulatif menggunakan qb100.finance.cumulative_returns
cum_ret = fin.cumulative_returns(daily_ret)
print("5 hari terakhir return kumulatif (%):")
(cum_ret * 100).round(2).tail()

## Visualisasi

In [ ]:
fig, ax = plt.subplots()
# ax.plot(...)
ax.set_title("Return harian & kumulatif")
# plotting.save_figure(fig, "day001_return-harian-kumulatif")
plt.show()

## Kesimpulan (3 poin)

1. ...
2. ...
3. ...

## Referensi

- ...